In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
from my_preprocessing_classes import CategoricalEncoder, reduce_mem_usage, TimeFeaturesEncoder, GroupAggregationTransformer, DropMissingFeatures

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━

In [5]:
import dagshub
import mlflow
dagshub.init(repo_owner='mkekn23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6ed775ed-30af-4cc6-82b8-56f16bfb279e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=d555b6403b47ab573bcf39f6b4321f4968a5bf2967c931922fdb2e82e12aad00




Accessing as mkekn23

Initialized MLflow to track repo "mkekn23/IEEE-CIS-Fraud-Detection"

Repository mkekn23/IEEE-CIS-Fraud-Detection initialized!

# Data Loading

In [6]:
df_identity= pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
df_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')

In [7]:
del df_transaction, df_identity

In [8]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

X = reduce_mem_usage(X)
y = y.astype('int8') 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Memory usage of dataframe is 1950.87 MB


/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow e

Memory usage after optimization is: 524.99 MB
Decreased by 73.1%


# Additional Feature Engineering

In [18]:
class GroupAggregateTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, groups, target_col):
        self.groups = groups
        self.target_col = target_col
        self.agg_maps = {}

    def fit(self, X, y=None):
        for group_col in self.groups:
            temp_group = X[group_col].astype('float32') if X[group_col].dtype == 'float16' else X[group_col]
            temp_target = X[self.target_col].astype('float32') if X[self.target_col].dtype == 'float16' else X[self.target_col]
            
            temp_df = pd.DataFrame({group_col: temp_group, self.target_col: temp_target})
            
           
            agg_data = temp_df.groupby(group_col)[self.target_col].agg(['mean', 'std']).reset_index()
            
            self.agg_maps[group_col] = agg_data
        return self

    def transform(self, X):
        X_out = X.copy()
        for group_col in self.groups:
            agg_data = self.agg_maps[group_col]
            
       
            if X_out[group_col].dtype == 'float16':
                X_out[group_col] = X_out[group_col].astype('float32')
            
            X_out = X_out.merge(agg_data, on=group_col, how='left')
            
            mean_col = f'{self.target_col}_to_mean_{group_col}'
            std_col = f'{self.target_col}_to_std_{group_col}'
            
            X_out.rename(columns={'mean': mean_col, 'std': std_col}, inplace=True)

            X_out[f'{self.target_col}_ratio_{group_col}'] = X_out[self.target_col].astype('float32') / X_out[mean_col]
            
        
            X_out[mean_col] = X_out[mean_col].astype('float16')
            X_out[std_col] = X_out[std_col].astype('float16')
            
        return X_out

# Training


In [9]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, average_precision_score

def calculate_fraud_metrics(y_true, y_pred, y_probs):
    return {
        "auc_roc": roc_auc_score(y_true, y_probs),
        "pr_auc": average_precision_score(y_true, y_probs), # მნიშვნელოვანია დისბალანსისას
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

import mlflow

def run_experiment(pipeline, run_name, params=None):
    with mlflow.start_run(run_name=run_name):
        if params:
            mlflow.log_params(params)
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_probs = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = calculate_fraud_metrics(y_test, y_pred, y_probs)
        

        mlflow.log_metrics(metrics)
        
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"Run '{run_name}' completed. AUC: {metrics['auc_roc']:.4f}")

In [10]:
from sklearn.pipeline import Pipeline

In [11]:
mlflow.set_experiment("LightGBM")

2026/05/03 14:07:51 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/f852c5ed7e60439aa4cf480b8b07233d', creation_time=1777817271520, experiment_id='4', last_update_time=1777817271520, lifecycle_stage='active', name='LightGBM', tags={}, trace_location=None, workspace='default'>

In [20]:
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer

lgbm_weak_params = {
    'n_estimators': 50,       
    'learning_rate': 0.5,    
    'num_leaves': 5,         
    'max_depth': 10,           
    'random_state': 42
}

lgbm_weak_pipeline = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),
    ('group_aggr', GroupAggregateTransformer(groups=['card1', 'addr1'], target_col='TransactionAmt')),
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()), 
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMClassifier(**lgbm_weak_params))
])

run_experiment(lgbm_weak_pipeline, "LGBM_Baseline")

[LightGBM] [Info] Number of positive: 16421, number of negative: 456011
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.416361 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39166
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 427
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034758 -> initscore=-3.323956
[LightGBM] [Info] Start training from score -3.323956


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/03 14:12:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:12:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'LGBM_Baseline' completed. AUC: 0.8290
🏃 View run LGBM_Baseline at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/45a1b44a61374ef6b6c61a44e2f63f46
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [21]:
from lightgbm import LGBMClassifier

lgbm_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 64,     
    'feature_fraction': 0.8,  
    'bagging_fraction': 0.8,  
    'bagging_freq': 5,
    'is_unbalance': True,    
    'random_state': 42,
    'n_jobs': -1
}

lgbm_pipeline = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),
    ('group_aggr', GroupAggregateTransformer(groups=['card1', 'addr1'], target_col='TransactionAmt')),
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()), 
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMClassifier(**lgbm_params))
])


run_experiment(lgbm_pipeline, "LightGBM_1")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Number of positive: 16421, number of negative: 456011
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.805421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[Lig

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


2026/05/03 14:17:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:17:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'LightGBM_1' completed. AUC: 0.9704
🏃 View run LightGBM_1 at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/aa34fd84b61d4d8f93c3827ebce317f4
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [22]:
# საწვრთნელი მონაცემების AUC
y_train_probs = lgbm_pipeline.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_probs)

# სატესტო მონაცემების AUC
y_test_probs = lgbm_pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_probs)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
Train AUC: 0.9986
Test AUC: 0.9704
Difference: 0.0282


In [23]:
from lightgbm import LGBMClassifier

lgbm_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 30,     
    'feature_fraction': 0.7,  
    'bagging_fraction': 0.8,  
    'bagging_freq': 5,
    'is_unbalance': True,    
    'random_state': 42,
    'n_jobs': -1
}

lgbm_pipeline = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),
    ('group_aggr', GroupAggregateTransformer(groups=['card1', 'addr1'], target_col='TransactionAmt')),
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()), 
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMClassifier(**lgbm_params))
])


run_experiment(lgbm_pipeline, "LightGBM_changed_params")

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Number of positive: 16421, number of negative: 456011
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.816222 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[Lig

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


2026/05/03 14:43:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 14:43:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'LightGBM_changed_params' completed. AUC: 0.9622
🏃 View run LightGBM_changed_params at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/4d6009ea0db644efab23d87c5a5d3491
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4


In [24]:
# საწვრთნელი მონაცემების AUC
y_train_probs = lgbm_pipeline.predict_proba(X_train)[:, 1]
train_auc = roc_auc_score(y_train, y_train_probs)

# სატესტო მონაცემების AUC
y_test_probs = lgbm_pipeline.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_test_probs)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.7, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
Train AUC: 0.9889
Test AUC: 0.9622
Difference: 0.0267
